# Fusion Weight Sweep on Colab

This notebook evaluates different fusion weights for:

`P_final = alpha * P_image + beta * P_text + gamma * P_prior`

It reuses your existing local model artifacts and dataset split, then exports CSV/JSON and figures.


In [ ]:
# 1) Install minimal dependencies (Colab)
!pip -q install pandas matplotlib tqdm


In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) Configure project paths
# Update PROJECT_ROOT to your own Drive path that contains this repository.
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyComputer/Code')
DATASET_ROOT = PROJECT_ROOT / 'Dataset/archive/SkinDisease'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts/multi_model_compare/efficientnet_b0'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts/fusion_weight_sweep'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('ARTIFACTS_DIR =', ARTIFACTS_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
assert DATASET_ROOT.exists(), f'DATASET_ROOT not found: {DATASET_ROOT}'
assert ARTIFACTS_DIR.exists(), f'ARTIFACTS_DIR not found: {ARTIFACTS_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 4) Import project modules
import sys
import json
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.local_hybrid import local_hybrid_infer, synthetic_symptom_for_label, topk_hit, f1_macro
from core.mock_engine import load_class_labels


In [ ]:
# 5) Build a fixed evaluation subset from test split
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
MAX_PER_CLASS = 5
SEED = 42

def collect_samples(dataset_root: Path, max_per_class: int = 5, seed: int = 42):
    rng = random.Random(seed)
    test_root = dataset_root / 'test'
    out = []
    for class_dir in sorted([p for p in test_root.iterdir() if p.is_dir()]):
        files = [p for p in sorted(class_dir.rglob('*')) if p.suffix.lower() in IMAGE_EXTS]
        if not files:
            continue
        picked = files if len(files) <= max_per_class else sorted(rng.sample(files, max_per_class))
        out.extend([(p, class_dir.name) for p in picked])
    return out

labels = load_class_labels(DATASET_ROOT)
samples = collect_samples(DATASET_ROOT, max_per_class=MAX_PER_CLASS, seed=SEED)

print('Num classes:', len(labels))
print('Num samples:', len(samples))
print('First sample:', samples[0] if samples else None)


In [ ]:
# 6) Define candidate fusion weights
# Constraint: alpha + beta + gamma = 1, gamma in [0.00, 0.20]
ALPHA_VALUES = [0.50, 0.60, 0.70, 0.80, 0.90]
BETA_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
GAMMA_MIN, GAMMA_MAX = 0.00, 0.20

candidates = []
for alpha in ALPHA_VALUES:
    for beta in BETA_VALUES:
        gamma = round(1.0 - alpha - beta, 2)
        if GAMMA_MIN <= gamma <= GAMMA_MAX:
            candidates.append((round(alpha, 2), round(beta, 2), gamma))

# Ensure default config is included
default_triplet = (0.70, 0.25, 0.05)
if default_triplet not in candidates:
    candidates.append(default_triplet)

candidates = sorted(list(set(candidates)))
print('Candidate count:', len(candidates))
print('Candidates:', candidates)


In [ ]:
# 7) Evaluation helpers
def evaluate_one_setting(alpha: float, beta: float, gamma: float):
    y_true, y_pred = [], []
    top3_hit_count = 0
    lat_ms = []
    source_counter = Counter()

    for path, true_label in samples:
        image_bytes = path.read_bytes()
        symptom = synthetic_symptom_for_label(true_label)

        t0 = time.perf_counter()
        try:
            result = local_hybrid_infer(
                image_bytes=image_bytes,
                symptom_text=symptom,
                labels=labels,
                artifacts_dir=ARTIFACTS_DIR,
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                mode='hybrid',
            )
            pred = result['primary_diagnosis']
            top3 = result.get('top3_candidates', [])
            source_counter[result.get('source', 'unknown')] += 1
        except Exception:
            pred = '__FAILED__'
            top3 = []
            source_counter['failed'] += 1

        lat_ms.append((time.perf_counter() - t0) * 1000.0)
        y_true.append(true_label)
        y_pred.append(pred)
        top3_hit_count += int(topk_hit(true_label, top3))

    top1 = sum(int(t == p) for t, p in zip(y_true, y_pred)) / max(len(y_true), 1)
    top3 = top3_hit_count / max(len(y_true), 1)
    macro_f1 = f1_macro(y_true, y_pred, labels)
    success_rate = 1.0 - (source_counter.get('failed', 0) / max(len(y_true), 1))

    return {
        'alpha': round(alpha, 2),
        'beta': round(beta, 2),
        'gamma': round(gamma, 2),
        'num_samples': len(y_true),
        'top1': round(top1, 4),
        'top3': round(top3, 4),
        'macro_f1': round(macro_f1, 4),
        'success_rate': round(success_rate, 4),
        'avg_latency_ms': round(float(np.mean(lat_ms)), 2),
        'source_local_hybrid': int(source_counter.get('local_hybrid', 0)),
        'source_failed': int(source_counter.get('failed', 0)),
    }

def evaluate_image_only_baseline():
    y_true, y_pred = [], []
    top3_hit_count = 0
    lat_ms = []

    for path, true_label in samples:
        image_bytes = path.read_bytes()
        t0 = time.perf_counter()
        result = local_hybrid_infer(
            image_bytes=image_bytes,
            symptom_text='',
            labels=labels,
            artifacts_dir=ARTIFACTS_DIR,
            mode='image_only',
        )
        lat_ms.append((time.perf_counter() - t0) * 1000.0)

        y_true.append(true_label)
        y_pred.append(result['primary_diagnosis'])
        top3_hit_count += int(topk_hit(true_label, result.get('top3_candidates', [])))

    return {
        'setting': 'image_only_baseline',
        'num_samples': len(y_true),
        'top1': round(sum(int(t == p) for t, p in zip(y_true, y_pred)) / len(y_true), 4),
        'top3': round(top3_hit_count / len(y_true), 4),
        'macro_f1': round(f1_macro(y_true, y_pred, labels), 4),
        'avg_latency_ms': round(float(np.mean(lat_ms)), 2),
    }


In [ ]:
# 8) Run sweep
rows = []
for alpha, beta, gamma in tqdm(candidates, desc='Weight sweep'):
    rows.append(evaluate_one_setting(alpha, beta, gamma))

df = pd.DataFrame(rows).sort_values(['top1', 'macro_f1', 'top3'], ascending=False).reset_index(drop=True)
baseline = evaluate_image_only_baseline()

print('Top 10 settings by Top-1:')
display(df.head(10))
print('Image-only baseline:')
print(baseline)


In [ ]:
# 9) Save CSV/JSON
csv_path = OUTPUT_DIR / 'fusion_weight_sweep.csv'
json_path = OUTPUT_DIR / 'fusion_weight_sweep.json'

df.to_csv(csv_path, index=False, encoding='utf-8')
payload = {
    'config': {
        'dataset_root': str(DATASET_ROOT),
        'artifacts_dir': str(ARTIFACTS_DIR),
        'max_per_class': MAX_PER_CLASS,
        'seed': SEED,
        'num_candidates': len(candidates),
    },
    'image_only_baseline': baseline,
    'results': df.to_dict(orient='records'),
}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print('Saved:', csv_path)
print('Saved:', json_path)


In [ ]:
# 10) Plot figures
# Figure A: Top-10 settings bar chart
topn = df.head(10).copy()
topn['setting'] = topn.apply(lambda r: f"a={r.alpha:.2f}, b={r.beta:.2f}, g={r.gamma:.2f}", axis=1)

fig, ax = plt.subplots(figsize=(14, 5.2))
x = np.arange(len(topn))
w = 0.25
ax.bar(x - w, topn['top1'], width=w, label='Top-1')
ax.bar(x, topn['top3'], width=w, label='Top-3')
ax.bar(x + w, topn['macro_f1'], width=w, label='Macro-F1')
ax.set_xticks(x)
ax.set_xticklabels(topn['setting'], rotation=20, ha='right')
ax.set_ylim(0, 1.0)
ax.set_title('Fusion Weight Sweep: Top-10 Settings')
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.3)
ax.legend()
fig.tight_layout()
fig_a = OUTPUT_DIR / 'fusion_weight_top10_bar.png'
fig.savefig(fig_a, dpi=180, bbox_inches='tight')
plt.show()

# Figure B: 3D scatter (alpha, beta, gamma -> top1 color)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(df['alpha'], df['beta'], df['gamma'], c=df['top1'], cmap='viridis', s=70)
ax.set_xlabel('alpha')
ax.set_ylabel('beta')
ax.set_zlabel('gamma')
ax.set_title('Fusion Weight Space (color=Top-1)')
cbar = fig.colorbar(sc, ax=ax, pad=0.12)
cbar.set_label('Top-1')
fig_b = OUTPUT_DIR / 'fusion_weight_3d_scatter.png'
fig.savefig(fig_b, dpi=180, bbox_inches='tight')
plt.show()

# Figure C: latency-performance scatter
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.scatter(df['avg_latency_ms'], df['top1'], s=70, alpha=0.8, label='weight settings')
ax.axhline(baseline['top1'], color='orange', linestyle='--', label='image_only top1')
ax.set_xlabel('Average latency (ms / sample)')
ax.set_ylabel('Top-1')
ax.set_title('Top-1 vs Latency')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig_c = OUTPUT_DIR / 'fusion_weight_latency_tradeoff.png'
fig.savefig(fig_c, dpi=180, bbox_inches='tight')
plt.show()

print('Saved figures:')
print('-', fig_a)
print('-', fig_b)
print('-', fig_c)


In [ ]:
# 11) Quick conclusion helper
best = df.iloc[0]
print('Best setting by Top-1:')
print({
    'alpha': float(best['alpha']),
    'beta': float(best['beta']),
    'gamma': float(best['gamma']),
    'top1': float(best['top1']),
    'top3': float(best['top3']),
    'macro_f1': float(best['macro_f1']),
    'avg_latency_ms': float(best['avg_latency_ms']),
})

print('Default setting (0.70, 0.25, 0.05):')
default_row = df[(df['alpha'] == 0.70) & (df['beta'] == 0.25) & (df['gamma'] == 0.05)]
display(default_row if not default_row.empty else 'Default setting not in candidate list')
